In [1]:
# ==========================================
# BÀI 3: KIỂM TRA MIỀN GIÁ TRỊ VÀ GIÁ TRỊ KHÔNG NHẤT QUÁN
# ==========================================
import pandas as pd
import numpy as np
from google.colab import drive

# 1. Mount Drive và Đọc dữ liệu thô từ sheet 'Du_lieu_tho'
drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/Nhap_Mon_Hoc_May/Buoi2/Du_lieu_thuc_hanh_chuan_bi_du_lieu.xlsx'

df_raw = pd.read_excel(file_path, sheet_name='Du_lieu_tho')
df = df_raw.copy()  # Tạo bản sao để kiểm tra, không sửa dữ liệu gốc

print("=== BẮT ĐẦU THỰC HIỆN BÀI 3 ===\n")

# ------------------------------------------
# Yêu cầu 10: Kiểm tra giá trị duy nhất và tần suất xuất hiện
# ------------------------------------------
print("--- 10. KIỂM TRA GIÁ TRỊ VÀ TẦN SUẤT CHO CÁC CỘT PHÂN LOẠI ---")
cols_to_check_cat = ['Category', 'Seller_City', 'Has_Image', 'Discount']

for col in cols_to_check_cat:
    print(f"\n* Tần suất giá trị (bao gồm NaN) của cột [{col}]:")
    print(df[col].value_counts(dropna=False))
    print("-" * 40)


# ------------------------------------------
# Yêu cầu 11 & 12: Chuyển đổi thử sang số và liệt kê các giá trị lỗi (không ghi đè)
# ------------------------------------------
print("\n--- 11 & 12. PHÁT HIỆN CÁC GIÁ TRỊ KHÔNG THỂ CHUYỂN THÀNH SỐ ---")
num_cols = ['Price_VND', 'Rating', 'Purchase_Count', 'Stock', 'Weight']

for col in num_cols:
    # Ep kiểu thử sang dạng số, nếu lỗi sẽ thành NaN
    temp_series = pd.to_numeric(df[col], errors='coerce')

    # Lấy các giá trị ban đầu mà KHÔNG PHẢI là NaN nhưng lại bị ép thành NaN (tức là chứa chuỗi/lỗi)
    invalid_mask = df[col].notnull() & temp_series.isnull()
    invalid_values = df.loc[invalid_mask, col].unique()

    print(f"* Cột [{col}] có {len(invalid_values)} giá trị chuỗi/lỗi không thể chuyển thành số: {invalid_values}")


# ------------------------------------------
# Yêu cầu 13: Phát hiện các giá trị ngoài miền hợp lệ
# ------------------------------------------
print("\n--- 13. PHÁT HIỆN DỮ LIỆU NGOÀI MIỀN HỢP LỆ ---")

# Tạo bản sao tạm ép kiểu số để thực hiện so sánh khoảng
df_temp_num = df.copy()
for col in ['Price_VND', 'Rating', 'Stock', 'Discount']:
    df_temp_num[col] = pd.to_numeric(df_temp_num[col], errors='coerce')

# Rating ngoài khoảng 1–5
invalid_rating = df[df_temp_num['Rating'].notnull() & ~df_temp_num['Rating'].between(1, 5)]
print(f"* Rating ngoài khoảng [1, 5]: {len(invalid_rating)} dòng")
if len(invalid_rating) > 0:
    display(invalid_rating[['Product_ID', 'Rating']])

# Price_VND không dương (<= 0)
invalid_price = df[df_temp_num['Price_VND'].notnull() & (df_temp_num['Price_VND'] <= 0)]
print(f"\n* Price_VND không dương (<= 0): {len(invalid_price)} dòng")
if len(invalid_price) > 0:
    display(invalid_price[['Product_ID', 'Price_VND']])

# Stock âm (< 0)
invalid_stock = df[df_temp_num['Stock'].notnull() & (df_temp_num['Stock'] < 0)]
print(f"\n* Stock âm (< 0): {len(invalid_stock)} dòng")
if len(invalid_stock) > 0:
    display(invalid_stock[['Product_ID', 'Stock']])

# Discount ngoài khoảng 0–1
invalid_discount = df[df_temp_num['Discount'].notnull() & ~df_temp_num['Discount'].between(0, 1)]
print(f"\n* Discount ngoài khoảng [0, 1]: {len(invalid_discount)} dòng")
if len(invalid_discount) > 0:
    display(invalid_discount[['Product_ID', 'Discount']])


# ------------------------------------------
# Yêu cầu 14: Chuyển thử Created_Date và liệt kê ngày không hợp lệ
# ------------------------------------------
print("\n--- 14. KIỂM TRA ĐỊNH DẠNG NGÀY THÁNG CỘT CREATED_DATE ---")

# Chuyển thử sang kiểu datetime
temp_date = pd.to_datetime(df['Created_Date'], errors='coerce')

# Tìm các dòng có ngày gốc không trống nhưng khi chuyển đổi bị lỗi (NaT)
invalid_date_mask = df['Created_Date'].notnull() & temp_date.isnull()
invalid_dates = df.loc[invalid_date_mask, 'Created_Date'].unique()

print(f"* Số giá trị ngày tháng không hợp lệ/lỗi định dạng: {len(invalid_dates)}")
print(f"* Các giá trị ngày lỗi liệt kê: {invalid_dates}")

Mounted at /content/drive
=== BẮT ĐẦU THỰC HIỆN BÀI 3 ===

--- 10. KIỂM TRA GIÁ TRỊ VÀ TẦN SUẤT CHO CÁC CỘT PHÂN LOẠI ---

* Tần suất giá trị (bao gồm NaN) của cột [Category]:
Category
Điện tử       48
Thời trang    44
Gia dụng      44
Sách          43
Mỹ phẩm       43
dien tu        1
ĐIỆN TỬ        1
Dien Tu        1
gia dung       1
THỜI TRANG     1
my pham        1
Name: count, dtype: int64
----------------------------------------

* Tần suất giá trị (bao gồm NaN) của cột [Seller_City]:
Seller_City
Hải Phòng          39
TP. Hồ Chí Minh    37
Bắc Ninh           33
Cần Thơ            28
Hà Nội             28
Đà Nẵng            24
Nam Định           24
NaN                 7
HN                  2
ha noi              1
HÀ NỘI              1
TP.HCM              1
HCM                 1
Da Nang             1
Hai Phong           1
Name: count, dtype: int64
----------------------------------------

* Tần suất giá trị (bao gồm NaN) của cột [Has_Image]:
Has_Image
Có              186
Không     

,Product_ID,Rating
74,SP0075,6.5
147,SP0148,-1



* Price_VND không dương (<= 0): 1 dòng


,Product_ID,Price_VND
142,SP0143,-120000



* Stock âm (< 0): 1 dòng


,Product_ID,Stock
39,SP0040,-15



* Discount ngoài khoảng [0, 1]: 2 dòng


,Product_ID,Discount
136,SP0137,1.5
194,SP0195,-0.1



--- 14. KIỂM TRA ĐỊNH DẠNG NGÀY THÁNG CỘT CREATED_DATE ---
* Số giá trị ngày tháng không hợp lệ/lỗi định dạng: 2
* Các giá trị ngày lỗi liệt kê: ['31/02/2026' 'tháng 5/2026']
